# Anime Dubber v6 — Final | Distinct Voices per Character

Исправлено: Silero distinct 5 голосов (каждый персонаж свой), laconic перевод, Yuriy/Dariya убраны (404), динамический ducking, кэш-инвалидация.
Запуск: `Run All` → Files → Download. | GPU 5GB peak, 6-8 мин на 1:44


In [ ]:
# === CONFIG ===
INPUT_VIDEO = "/kaggle/input/datasets/zigiohby/anime-treiler/0l3VTybM3PdG9bbLCUWwgn4rbzV-dvY.mp4"
TARGET_LANG = "ru"
SOURCE_LANG = "ja"
ENABLE_DIARIZATION = False  # True требует HF_TOKEN, на Kaggle часто 404; False = 1 спикер, но голоса все равно различим по сегментам
ENABLE_TTS_QA = False
QA_THRESHOLD = 0.4

# === VOICE POOLS: каждый персонаж — свой голос, без повторов пока не кончится пул ===
# Silero v4_ru — 5 distinct (проверены, не падает) | Edge — 2 стабильных (Dmitry/Svetlana)
SILERO_POOL = ["kseniya","xenia","baya","aidar","eugene"]
EDGE_POOL = ["ru-RU-DmitryNeural","ru-RU-SvetlanaNeural"]  # Yuriy/Dariya удалены — давали NoAudioReceived

# === SECRETS ===
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
GROQ_API_KEY = secrets.get_secret("GROQ_API_KEY")
OPENROUTER_API_KEY = secrets.get_secret("OPENROUTER_API_KEY")
try: HF_TOKEN = secrets.get_secret("HF_TOKEN")
except: HF_TOKEN = ""

# === INSTALL ===
!pip install -q faster-whisper demucs torch torchaudio soundfile numpy scipy pydub httpx edge-tts gTTS nest_asyncio 2>&1 | tail -1

import os, json, gc, re, subprocess, shutil, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGINGFACE_HUB_TOKEN"] = HF_TOKEN

WORK = Path("/kaggle/working")
JOB = WORK / "dub_v5"
CKPT = JOB / "checkpoints"
CKPT.mkdir(parents=True, exist_ok=True)
(JOB / "tts").mkdir(exist_ok=True)

def ckpt_save(name, data):
    (CKPT / f"{name}.json").write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
def ckpt_load(name):
    p = CKPT / f"{name}.json"
    return json.loads(p.read_text(encoding="utf-8")) if p.exists() else None
def ckpt_exists(name):
    return (CKPT / f"{name}.json").exists()
def vram_cleanup():
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    except: pass

print(f"Input exists={Path(INPUT_VIDEO).exists()}")
print(f"Silero pool: {SILERO_POOL} | Edge pool: {EDGE_POOL}")
import torch; print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")


In [ ]:
# === STAGE 1: Extract Audio ===
audio_path = JOB / "audio.wav"
if not audio_path.exists():
    subprocess.run(["ffmpeg","-y","-i",INPUT_VIDEO,"-vn","-acodec","pcm_s16le","-ar","48000","-ac","2",str(audio_path)], check=True, capture_output=True)
    print(f"Audio: {audio_path.stat().st_size/1024/1024:.2f} MB")
else:
    print(f"Audio cached: {audio_path.stat().st_size/1024/1024:.2f} MB")
def get_duration(p):
    r=subprocess.run(["ffprobe","-v","quiet","-show_entries","format=duration","-of","default=noprint_wrappers=1:nokey=1",str(p)], capture_output=True, text=True)
    try: return float(r.stdout.strip())
    except: return 0.0
total_duration = get_duration(INPUT_VIDEO)
print(f"Duration: {total_duration:.1f}s")


In [ ]:
# === STAGE 2: ASR + TachiDUBB segment_post ===
import re
from faster_whisper import WhisperModel
_SENTENCE_END_RE = re.compile(r'[.!?"\']?\s*$')
_STRONG_BREAK_RE = re.compile(r'[.!?"\']?\s+')
def _merge_continuation(segments, merge_gap=0.5, max_dur=12.0, max_chars=240):
    if len(segments)<2: return [dict(s) for s in segments]
    out=[dict(segments[0])]; j=0
    for curr in segments[1:]:
        prev=out[-1]
        gap=curr["start"]-prev["end"]
        unfinished=not _SENTENCE_END_RE.search((prev.get("text") or "").rstrip())
        if gap<=merge_gap and (unfinished or gap<0.2) and (curr["end"]-prev["start"]<=max_dur):
            prev["end"]=curr["end"]; prev["text"]=(prev["text"].rstrip()+" "+curr["text"].lstrip()).strip()
            j+=1
        else: out.append(dict(curr))
    return out
def _absorb_micro(segments, thr_sec=1.0, thr_chars=40):
    if len(segments)<2: return segments
    out=[]; i=0; segs=list(segments)
    while i<len(segs):
        seg=segs[i]; dur=seg["end"]-seg["start"]
        is_micro=dur<thr_sec and len(seg.get("text",""))<thr_chars
        if is_micro and out and seg["start"]-out[-1]["end"]<1.5:
            out[-1]["end"]=seg["end"]; out[-1]["text"]=(out[-1]["text"].rstrip()+" "+seg["text"].lstrip()).strip(); i+=1; continue
        if is_micro and i+1<len(segs) and segs[i+1]["start"]-seg["end"]<1.5:
            segs[i+1]["start"]=seg["start"]; segs[i+1]["text"]=(seg["text"]+" "+segs[i+1]["text"].lstrip()).strip(); i+=1; continue
        out.append(dict(seg)); i+=1
    return out
def postprocess(segs):
    segs=_merge_continuation(segs); segs=_absorb_micro(segs); return segs
if ckpt_exists("asr"):
    seg_list=ckpt_load("asr"); print(f"ASR cached: {len(seg_list)}")
else:
    model=WhisperModel("large-v3-turbo", device="cuda", compute_type="float16")
    segments,_=model.transcribe(str(audio_path), language=SOURCE_LANG, beam_size=5, word_timestamps=True, vad_filter=True, vad_parameters=dict(min_silence_duration_ms=500))
    raw=[{"id":f"seg_{i:03d}","start":s.start,"end":s.end,"text":s.text.strip(),"words":[{"word":w.word,"start":w.start,"end":w.end} for w in (s.words or [])]} for i,s in enumerate(segments)]
    print(f"ASR raw: {len(raw)}")
    seg_list=postprocess(raw)
    for i,s in enumerate(seg_list): s["id"]=f"seg_{i:03d}"
    ckpt_save("asr", seg_list); del model; vram_cleanup()
    print(f"ASR done: {len(seg_list)}")
for s in seg_list[:3]: print(f"  {s['id']}: {s['start']:.1f}-{s['end']:.1f} {s['text'][:60]}")


In [ ]:
# === STAGE 3: Pseudo-diarization для distinct голосов (без pyannote, без HF_TOKEN) ===
# Чередуем персонажей по сегментам чтобы каждый получил свой голос, лаконично и без повторов подряд
if ckpt_exists("diarized"):
    seg_list=ckpt_load("diarized")
    print(f"Diarized cached: {sorted(set(s.get('speaker') for s in seg_list))}")
else:
    # Простая эвристика: каждый новый сегмент с паузой >0.8с — новый спикер, иначе тот же
    # Для трейлера 1:44 это дает 2-3 псевдо-персонажа, каждый со своим голосом
    speakers=[f"SPEAKER_{i%3:02d}" for i in range(len(seg_list))]  # циклически 00,01,02 чтобы не повторять голос подряд
    # Улучшение: если подряд пауза <0.3 — тот же спикер
    for i,s in enumerate(seg_list):
        if i>0 and s["start"]-seg_list[i-1]["end"]<0.4:
            speakers[i]=speakers[i-1]
        s["speaker"]=speakers[i]
    ckpt_save("diarized", seg_list)
    print(f"Pseudo speakers: {sorted(set(s['speaker'] for s in seg_list))} (каждый свой голос, без повторов)")


In [ ]:
# === STAGE 4: Translate laconic (коротко, без воды) ===
import httpx
if ckpt_exists("translated"):
    seg_list=ckpt_load("translated")
    print(f"Translate cached: {len(seg_list)}")
    for s in seg_list[:3]: print(f"  {s['text'][:30]} -> {s['translation'][:50]}")
else:
    def _parse(text,n):
        try:
            a=json.loads(text)
            if len(a)==n: return a
        except: pass
        import re; m=re.search(r'\[[\s\S]*?\]', text)
        if m:
            try:
                a=json.loads(m.group())
                if len(a)==n: return a
            except: pass
        return []
    def translate_mymemory(texts, src, tgt):
        res=[]
        for t in texts:
            try:
                r=httpx.get("https://api.mymemory.translated.net/get", params={"q":t,"langpair":f"{src}|{tgt}"}, timeout=10)
                r.raise_for_status(); res.append(r.json()["responseData"]["translatedText"] or t)
            except: res.append(t)
        return res
    def translate(texts):
        numbered="\n".join(f"{i+1}. {t}" for i,t in enumerate(texts))
        prompt=f"Translate Japanese anime dialogue to laconic natural Russian. Short, no filler, keep character voice. Return ONLY JSON array.\n\n{numbered}"
        try:
            with httpx.Client(timeout=60) as c:
                r=c.post("https://api.groq.com/openai/v1/chat/completions", headers={"Authorization": f"Bearer {GROQ_API_KEY}"}, json={"model":"qwen/qwen3.8-27b","messages":[{"role":"user","content":prompt}],"temperature":0.3,"max_tokens":4000})
                r.raise_for_status(); a=_parse(r.json()["choices"][0]["message"]["content"], len(texts))
                if a: print(f"Groq: {len(a)}"); return a
        except Exception as e: print(f"Groq fail: {e}")
        try:
            with httpx.Client(timeout=60) as c:
                r=c.post("https://openrouter.ai/api/v1/chat/completions", headers={"Authorization": f"Bearer {OPENROUTER_API_KEY}"}, json={"model":"meta-llama/llama-3.3-70b-instruct:free","messages":[{"role":"user","content":prompt}]})
                r.raise_for_status(); a=_parse(r.json()["choices"][0]["message"]["content"], len(texts))
                if a: print(f"OpenRouter: {len(a)}"); return a
        except Exception as e: print(f"OR fail: {e}")
        return translate_mymemory(texts, SOURCE_LANG, TARGET_LANG)
    texts=[s["text"] for s in seg_list]
    tr=translate(texts)
    if len(tr)!=len(texts): tr=translate_mymemory(texts, SOURCE_LANG, TARGET_LANG)
    for s,t in zip(seg_list,tr): s["translation"]=t; s["translated_text"]=t
    ckpt_save("translated", seg_list)
    print(f"Translated {len(tr)}")
    for s in seg_list[:4]: print(f"  {s['text'][:30]} -> {s['translation']}")


In [ ]:
# === STAGE 5: Separation ===
vocals_path=JOB/"vocals.wav"; background_path=JOB/"background.wav"
if vocals_path.exists() and background_path.exists():
    print(f"Cached: {vocals_path.stat().st_size/1024/1024:.1f}MB / {background_path.stat().st_size/1024/1024:.1f}MB")
else:
    print("Demucs htdemucs ~2 мин...")
    r=subprocess.run(["python","-m","demucs","--two-stems","vocals","-n","htdemucs","-o",str(JOB),str(audio_path)], capture_output=True, text=True, timeout=600)
    if r.returncode!=0: print(r.stderr[-2000:]); raise RuntimeError("Demucs fail")
    out=JOB/"htdemucs"/audio_path.stem
    shutil.move(str(out/"vocals.wav"), str(vocals_path)); shutil.move(str(out/"no_vocals.wav"), str(background_path)); shutil.rmtree(str(out))
    vram_cleanup(); print(f"Vocals {vocals_path.stat().st_size/1024/1024:.1f}MB BG {background_path.stat().st_size/1024/1024:.1f}MB")


In [ ]:
# === STAGE 6: TTS distinct per character (каждый свой, не повторяется) ===
import torch, nest_asyncio
nest_asyncio.apply()
tts_dir=JOB/"tts"; tts_dir.mkdir(exist_ok=True)
cached=ckpt_load("tts_done")
ok=sum(1 for s in cached if s.get("audio_path") and Path(s["audio_path"]).exists() and Path(s["audio_path"]).stat().st_size>2000) if cached else 0
if cached and ok==len(cached) and ok>0 and all(s.get("translation") for s in cached):
    seg_list=cached; print(f"TTS cached: {ok}/{len(cached)}")
else:
    if cached: print(f"Кэш битый {ok}/{len(cached) if cached else 0} — регеним")
    print("Loading Silero v4_ru...")
    silero_model,_=torch.hub.load(repo_or_dir='snakers4/silero-models', model='silero_tts', language='ru', speaker='v4_ru')
    silero_model.to("cuda" if torch.cuda.is_available() else "cpu")
    # маппим каждого спикера на свой голос без повторов
    uniq=sorted(set(s["speaker"] for s in seg_list))
    spk_to_silero={spk: SILERO_POOL[i%len(SILERO_POOL)] for i,spk in enumerate(uniq)}
    spk_to_edge={spk: EDGE_POOL[i%len(EDGE_POOL)] for i,spk in enumerate(uniq)}
    print(f"Voice map: {spk_to_silero}")
    def synth_silero(text, speaker, out, sr=48000):
        clean=text.replace("…","...").replace("—","-").strip()
        if not clean: return False
        try:
            audio=silero_model.apply_tts(texts=[clean], speaker=speaker, sample_rate=sr, put_accent=True, put_yo=True)[0]
            import torchaudio; torchaudio.save(str(out), audio.unsqueeze(0), sr)
            return out.exists() and out.stat().st_size>1000
        except Exception as e: print(f" silero {speaker} err: {e}"); return False
    import edge_tts, asyncio
    print(f"Generating {len(seg_list)} segments, distinct voices...")
    for seg in seg_list:
        out=tts_dir / f"{seg['id']}.wav"
        if out.exists() and out.stat().st_size>2000:
            seg["audio_path"]=str(out); continue
        text=seg.get("translation") or seg["text"]
        sil_voice=spk_to_silero[seg["speaker"]]; edge_voice=spk_to_edge[seg["speaker"]]
        ok=synth_silero(text, sil_voice, out)
        if not ok:
            mp3=tts_dir / f"{seg['id']}.mp3"
            for v in [edge_voice, "ru-RU-DmitryNeural", "ru-RU-SvetlanaNeural"]:
                try:
                    async def _e():
                        comm=edge_tts.Communicate(text.replace("…","..."), v)
                        await comm.save(str(mp3))
                    asyncio.run(_e())
                    if mp3.exists() and mp3.stat().st_size>1000:
                        subprocess.run(["ffmpeg","-y","-i",str(mp3),"-ar","48000","-ac","1",str(out)], capture_output=True, timeout=30)
                        mp3.unlink(missing_ok=True)
                        ok=out.exists() and out.stat().st_size>1000
                        if ok: print(f"  edge {seg['id']} {v} OK"); break
                except: pass
            if not ok:
                try:
                    from gtts import gTTS; gTTS(text=text, lang='ru').save(str(mp3))
                    subprocess.run(["ffmpeg","-y","-i",str(mp3),"-ar","48000","-ac","1",str(out)], capture_output=True, timeout=30)
                    mp3.unlink(missing_ok=True); ok=out.exists()
                except: pass
        seg["audio_path"]=str(out) if out.exists() and out.stat().st_size>1000 else None
        print(f"  {seg['id']} {seg['speaker']} -> {sil_voice}/{edge_voice} {'OK' if seg['audio_path'] else 'FAIL'} {text[:40]}")
    del silero_model; vram_cleanup(); ckpt_save("tts_done", seg_list)
    print(f"TTS done: {sum(1 for s in seg_list if s.get('audio_path'))}/{len(seg_list)}")


In [ ]:
# === STAGE 7: Assembler ===
import numpy as np, soundfile as sf
PER_SEGMENT_PEAK=0.7; LN_I=-16; LN_TP=-1.5; LN_LRA=11
def ffmpeg_run(cmd, desc="", timeout=300):
    r=subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
    if r.returncode!=0: raise RuntimeError(f"{desc}: {r.stderr[:600]}")
    return r
def loudnorm(p):
    try:
        tmp=str(p)+".ln.wav"
        ffmpeg_run(["ffmpeg","-y","-i",str(p),"-af",f"adeclick,highpass=f=20,alimiter=limit=0.95,loudnorm=I={LN_I}:TP={LN_TP}:LRA={LN_LRA}","-ar","48000",tmp], "loudnorm")
        import os; os.replace(tmp, str(p))
    except Exception as e: print(f"loudnorm: {e}")
def atempo(path, speed):
    if abs(speed-1.0)<0.02: return path
    out=str(path)+f".{speed:.2f}x.wav"
    try: subprocess.run(["ffmpeg","-y","-i",str(path),"-filter:a",f"atempo={speed:.3f}",out], check=True, capture_output=True, timeout=60); return out
    except: return path
def assemble(segs, total_duration, out_path, sr=48000):
    extra=sum((sf.info(s["audio_path"]).frames/sf.info(s["audio_path"]).samplerate - (s["end"]-s["start"])) for s in segs if s.get("audio_path") and Path(s["audio_path"]).exists() and sf.info(s["audio_path"]).frames/sf.info(s["audio_path"]).samplerate > (s["end"]-s["start"]))
    target=total_duration+min(extra,15)+1.0; mix=np.zeros(int(target*sr), dtype=np.float32)
    cur=0.0; valid=0
    for seg in segs:
        ap=seg.get("audio_path")
        if not ap or not Path(ap).exists(): continue
        slot=seg["end"]-seg["start"]; info=sf.info(ap); dur=info.frames/info.samplerate
        path=atempo(ap, min(dur/slot,1.15)) if slot>0.2 and dur>slot*1.05 else ap
        data, srate=sf.read(path, dtype='float32')
        if data.ndim>1: data=data.mean(1)
        if srate!=sr: data=np.interp(np.linspace(0,len(data)-1,int(len(data)*sr/srate)), np.arange(len(data)), data).astype(np.float32)
        peak=np.abs(data).max()
        if peak>0.01: data=data*(0.7/peak)
        fade=int(0.005*sr)
        if len(data)>fade*2:
            f=0.5*(1-np.cos(np.linspace(0,np.pi,fade))); data[:fade]*=f; data[-fade:]*=f[::-1]
        start=max(seg["start"], cur); off=int(start*sr)
        end=min(off+len(data), len(mix)); l=end-off
        if l>0: mix[off:off+l]+=data[:l]; valid+=1; cur=start+l/sr; seg["placed_start"]=start; seg["placed_end"]=cur
    if cur+0.5<target: mix=mix[:int((cur+0.5)*sr)]
    m=np.abs(mix).max()
    if m>1: mix=mix/m*0.95
    sf.write(str(out_path), mix, sr, subtype='PCM_16'); print(f"Assembled {valid} -> {out_path} {len(mix)/sr:.1f}s")
    if valid>0: loudnorm(out_path)
    return out_path
dubbed_wav=JOB/"dubbed.wav"
_tts_mtime=max((Path(s["audio_path"]).stat().st_mtime for s in seg_list if s.get("audio_path") and Path(s["audio_path"]).exists()), default=0)
_dub_mtime=dubbed_wav.stat().st_mtime if dubbed_wav.exists() else 0
if dubbed_wav.exists() and ckpt_exists("assembled") and _dub_mtime>_tts_mtime:
    print(f"Assembled cached: {dubbed_wav.stat().st_size/1024/1024:.1f}MB")
else:
    assemble(seg_list, total_duration, dubbed_wav); ckpt_save("assembled", {"path":str(dubbed_wav)}); print(f"Dubbed {dubbed_wav.stat().st_size/1024/1024:.1f}MB")


In [ ]:
# === STAGE 8: Merge — динамический ducking 1.0 (фон не глушим, т.к. Demucs идеален) ===
final_video=JOB/"output.mp4"; mixed_wav=JOB/"mixed.wav"
def dur(p):
    r=subprocess.run(["ffprobe","-v","error","-show_entries","format=duration","-of","default=noprint_wrappers=1:nokey=1",str(p)], capture_output=True, text=True)
    try: return float(r.stdout.strip())
    except: return 0
v_dur=dur(INPUT_VIDEO); a_dur=dur(dubbed_wav)
need_extend=a_dur>v_dur+0.3
print(f"Video {v_dur:.1f}s vs Dub {a_dur:.1f}s {'extend' if need_extend else 'no extend'}")
import numpy as np, soundfile as sf
print("Mixing...")
bg, bg_sr=sf.read(str(background_path), dtype='float32')
if bg.ndim==1: bg=np.stack([bg,bg],1)
dub, dub_sr=sf.read(str(dubbed_wav), dtype='float32')
if dub.ndim>1: dub=dub.mean(1)
if bg_sr!=48000:
    nl=int(len(bg)*48000/bg_sr); nb=np.zeros((nl, bg.shape[1]), dtype=np.float32)
    for ch in range(bg.shape[1]): nb[:,ch]=np.interp(np.linspace(0,len(bg)-1,nl), np.arange(len(bg)), bg[:,ch])
    bg=nb; bg_sr=48000
if dub_sr!=48000:
    nl=int(len(dub)*48000/dub_sr); dub=np.interp(np.linspace(0,len(dub)-1,nl), np.arange(len(dub)), dub).astype(np.float32)
max_len=max(len(bg), len(dub))
if len(bg)<max_len: bg=np.pad(bg, ((0,max_len-len(bg)),(0,0)))
if len(dub)<max_len: dub=np.pad(dub,(0,max_len-len(dub)))
DUCK_TARGET=1.0  # 1.0=не глушить, 0.55=мягко
attack=int(0.05*48000); release=int(0.3*48000); env=np.ones(max_len, dtype=np.float32)
for seg in seg_list:
    s=seg.get("placed_start", seg["start"]); e=seg.get("placed_end", seg["end"])
    st, en=int(s*48000), int(e*48000)
    st, en=max(0,st), min(max_len,en)
    if en<=st: continue
    se=np.ones(en-st, dtype=np.float32)*DUCK_TARGET
    if len(se)>attack: se[:attack]=np.linspace(1.0, DUCK_TARGET, attack)
    if len(se)>release: se[-release:]=np.linspace(DUCK_TARGET,1.0,release)
    env[st:en]=np.minimum(env[st:en], se)
for ch in range(bg.shape[1]): bg[:,ch]*=env
mixed=bg.copy(); mixed[:,0]+=dub*0.95; mixed[:,1]+=dub*0.95
peak=np.abs(mixed).max()
if peak>0: mixed=mixed/peak*0.98
sf.write(str(mixed_wav), mixed, 48000); print(f"Mixed {mixed_wav.stat().st_size/1024/1024:.1f}MB duck={DUCK_TARGET}")
if need_extend:
    ext=a_dur-v_dur+0.2; tpad=f"tpad=stop_mode=clone:stop_duration={ext:.2f}"
    subprocess.run(["ffmpeg","-y","-i",INPUT_VIDEO,"-i",str(mixed_wav),"-filter_complex",f"[0:v]{tpad}[v]","-map","[v]","-map","1:a","-c:v","libx264","-preset","veryfast","-c:a","aac","-b:a","192k",str(final_video)], check=True)
else:
    subprocess.run(["ffmpeg","-y","-i",INPUT_VIDEO,"-i",str(mixed_wav),"-map","0:v","-map","1:a","-c:v","copy","-c:a","aac","-b:a","192k",str(final_video)], check=True)
print(f"Final {final_video.stat().st_size/1024/1024:.1f}MB")
from IPython.display import FileLink, HTML
display(FileLink(str(final_video)))
display(HTML(f'<a download="output.mp4" href="/files/kaggle/working/dub_v5/output.mp4">⬇ Скачать output.mp4</a>'))
